In [1]:
from dotenv import load_dotenv

# 加载环境变量
load_dotenv()

True

# 官方示例 1

代码：[Raudaschl/rag-fusion](https://github.com/Raudaschl/rag-fusion)

### 生成多个查询语句

In [2]:
from openai import OpenAI
client = OpenAI()

# 使用 OpenAI 的 ChatGPT 生成查询的函数
def generate_queries_chatgpt(original_query):
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "你是一个有帮助的助手，会根据单个输入查询生成多个搜索查询。"},
            {"role": "user", "content": f"生成与以下内容相关的多个搜索查询：{original_query}"},
            {"role": "user", "content": "输出（4 个查询）："}
        ]
    )
    generated_queries = response.choices[0].message.content.strip().split("\n")
    return generated_queries

In [3]:
original_query = "气候变化的影响"
generated_queries = generate_queries_chatgpt(original_query)

In [4]:
generated_queries

['1. 气候变化对生态系统的影响', '2. 气候变化如何影响农业', '3. 气候变化对海洋生物的影响', '4. 气候变化对人类健康的影响']

### 模拟向量搜索

该函数模拟向量搜索，从所有文档中随机选择2到5个文档，并为它们分配0.7到0.9之间的随机分数。然后返回这些文档及其分数，按分数降序排列。

In [5]:
import random

# 模拟向量搜索的函数，返回随机分数
def vector_search(query, all_documents):
    available_docs = list(all_documents.keys())
    random.shuffle(available_docs)
    selected_docs = available_docs[:random.randint(2, 5)]
    scores = {doc: round(random.uniform(0.7, 0.9), 2) for doc in selected_docs}
    return {doc: score for doc, score in sorted(scores.items(), key=lambda x: x[1], reverse=True)}

In [6]:
# 预定义的文档集合（通常这些会来自你的搜索数据库）
all_documents = {
    "文档1": "气候变化和经济影响。",
    "文档2": "气候变化对公共健康的担忧。",
    "文档3": "气候变化：社会视角。",
    "文档4": "应对气候变化的技术解决方案。",
    "文档5": "应对气候变化所需的政策变更。",
    "文档6": "气候变化对生物多样性的影响。",
    "文档7": "气候变化：科学与模型。",
    "文档8": "全球变暖：气候变化的一个子集。",
    "文档9": "气候变化如何影响日常天气。",
    "文档10": "气候变化活动的历史。"
}

In [7]:
all_results = {}
for query in generated_queries:
    search_results = vector_search(query, all_documents)
    all_results[query] = search_results

In [8]:
all_results

{'1. 气候变化对生态系统的影响': {'文档9': 0.72, '文档7': 0.72, '文档2': 0.7},
 '2. 气候变化如何影响农业': {'文档5': 0.8,
  '文档2': 0.77,
  '文档7': 0.77,
  '文档10': 0.77,
  '文档1': 0.73},
 '3. 气候变化对海洋生物的影响': {'文档10': 0.7, '文档2': 0.7},
 '4. 气候变化对人类健康的影响': {'文档4': 0.87,
  '文档1': 0.84,
  '文档7': 0.78,
  '文档2': 0.76,
  '文档3': 0.75}}

### Reciprocal Rank Fusion 算法

这个函数对所有查询的结果进行融合，并重新排序

In [9]:
# Reciprocal Rank Fusion 算法
def reciprocal_rank_fusion(search_results_dict, k=60):
    fused_scores = {}
    print("初始的单个搜索结果排名：")
    for query, doc_scores in search_results_dict.items():
        print(f"查询 '{query}' 的结果：{doc_scores}")
        
    for query, doc_scores in search_results_dict.items():
        for rank, (doc, score) in enumerate(sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)):
            if doc not in fused_scores:
                fused_scores[doc] = 0
            previous_score = fused_scores[doc]
            fused_scores[doc] += 1 / (rank + k)
            print(f"基于查询 '{query}' 中的排名 {rank}，更新文档 {doc} 的得分从 {previous_score} 到 {fused_scores[doc]}")

    reranked_results = {doc: score for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)}
    print("最终重新排序的结果：", reranked_results)
    return reranked_results

In [10]:
reranked_results = reciprocal_rank_fusion(all_results)

初始的单个搜索结果排名：
查询 '1. 气候变化对生态系统的影响' 的结果：{'文档9': 0.72, '文档7': 0.72, '文档2': 0.7}
查询 '2. 气候变化如何影响农业' 的结果：{'文档5': 0.8, '文档2': 0.77, '文档7': 0.77, '文档10': 0.77, '文档1': 0.73}
查询 '3. 气候变化对海洋生物的影响' 的结果：{'文档10': 0.7, '文档2': 0.7}
查询 '4. 气候变化对人类健康的影响' 的结果：{'文档4': 0.87, '文档1': 0.84, '文档7': 0.78, '文档2': 0.76, '文档3': 0.75}
基于查询 '1. 气候变化对生态系统的影响' 中的排名 0，更新文档 文档9 的得分从 0 到 0.016666666666666666
基于查询 '1. 气候变化对生态系统的影响' 中的排名 1，更新文档 文档7 的得分从 0 到 0.01639344262295082
基于查询 '1. 气候变化对生态系统的影响' 中的排名 2，更新文档 文档2 的得分从 0 到 0.016129032258064516
基于查询 '2. 气候变化如何影响农业' 中的排名 0，更新文档 文档5 的得分从 0 到 0.016666666666666666
基于查询 '2. 气候变化如何影响农业' 中的排名 1，更新文档 文档2 的得分从 0.016129032258064516 到 0.03252247488101534
基于查询 '2. 气候变化如何影响农业' 中的排名 2，更新文档 文档7 的得分从 0.01639344262295082 到 0.03252247488101534
基于查询 '2. 气候变化如何影响农业' 中的排名 3，更新文档 文档10 的得分从 0 到 0.015873015873015872
基于查询 '2. 气候变化如何影响农业' 中的排名 4，更新文档 文档1 的得分从 0 到 0.015625
基于查询 '3. 气候变化对海洋生物的影响' 中的排名 0，更新文档 文档10 的得分从 0.015873015873015872 到 0.032539682539682535
基于查询 '3. 气候变化对海洋生物的影响' 中的排名 1，更新文档 文档2

## 生成最终输出

In [11]:
# 模拟生成输出的函数
def generate_output(reranked_results, queries):
    return f"基于 {queries} 和重新排序的文档生成的最终输出：{list(reranked_results.keys())}"

In [12]:
final_output = generate_output(reranked_results, generated_queries)

print(final_output)

基于 ['1. 气候变化对生态系统的影响', '2. 气候变化如何影响农业', '3. 气候变化对海洋生物的影响', '4. 气候变化对人类健康的影响'] 和重新排序的文档生成的最终输出：['文档2', '文档7', '文档10', '文档1', '文档9', '文档5', '文档4', '文档3']


# 官方示例 2

LangChain Cookbook：[RAG Fusion](https://github.com/langchain-ai/langchain/blob/master/cookbook/rag_fusion.ipynb?ref=blog.langchain.dev)

## 生成多个查询语句

### prompt

In [13]:
# from langchain import hub

# # https://smith.langchain.com/hub/langchain-ai/rag-fusion-query-generation
# prompt = hub.pull("langchain-ai/rag-fusion-query-generation")

In [14]:
# from langchain.prompts import ChatPromptTemplate

# prompt = ChatPromptTemplate.from_messages([
#     ("system", "You are a helpful assistant that generates multiple search queries based on a single input query."),
#     ("user", "Generate multiple search queries related to: {original_query}"),
#     ("user", "OUTPUT (4 queries):")
# ])

In [15]:
from langchain.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个有帮助的助手，会根据单个输入查询生成多个搜索查询。"),
    ("user", "生成与以下内容相关的多个搜索查询：{original_query}"),
    ("user", "输出（4 个查询）：")
])

### 查询生成链

In [16]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

In [17]:
generate_queries = (
    prompt | ChatOpenAI(temperature=0) | StrOutputParser() | (lambda x: x.split("\n"))
)

In [18]:
original_query = "气候变化的影响"

In [19]:
generate_queries.invoke({"original_query": original_query})

['1. 气候变化对农业的影响', '2. 气候变化对海洋生态系统的影响', '3. 气候变化对林业的影响', '4. 气候变化对人类健康的影响']

## 向量搜索

### data

In [20]:
# 预定义的文档集合（通常这些会来自你的搜索数据库）
all_documents = {
    "文档1": "气候变化和经济影响。",
    "文档2": "气候变化对公共健康的担忧。",
    "文档3": "气候变化：社会视角。",
    "文档4": "应对气候变化的技术解决方案。",
    "文档5": "应对气候变化所需的政策变更。",
    "文档6": "气候变化对生物多样性的影响。",
    "文档7": "气候变化：科学与模型。",
    "文档8": "全球变暖：气候变化的一个子集。",
    "文档9": "气候变化如何影响日常天气。",
    "文档10": "气候变化活动的历史。"
}

In [21]:
list(all_documents.values())

['气候变化和经济影响。',
 '气候变化对公共健康的担忧。',
 '气候变化：社会视角。',
 '应对气候变化的技术解决方案。',
 '应对气候变化所需的政策变更。',
 '气候变化对生物多样性的影响。',
 '气候变化：科学与模型。',
 '全球变暖：气候变化的一个子集。',
 '气候变化如何影响日常天气。',
 '气候变化活动的历史。']

### retriever

In [22]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()

In [23]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_texts(
    list(all_documents.values()), embeddings
)

In [24]:
retriever = vectorstore.as_retriever()

In [25]:
# test

retriever.invoke(original_query)

[Document(page_content='气候变化和经济影响。'),
 Document(page_content='气候变化如何影响日常天气。'),
 Document(page_content='气候变化对生物多样性的影响。'),
 Document(page_content='气候变化对公共健康的担忧。')]

### 检索链

每个生成的新query都去检索文档

In [26]:
chain = generate_queries | retriever.map()

In [27]:
chain.invoke({"original_query": original_query})

[[Document(page_content='气候变化和经济影响。'),
  Document(page_content='气候变化如何影响日常天气。'),
  Document(page_content='气候变化对生物多样性的影响。'),
  Document(page_content='气候变化：社会视角。')],
 [Document(page_content='气候变化对生物多样性的影响。'),
  Document(page_content='气候变化和经济影响。'),
  Document(page_content='气候变化如何影响日常天气。'),
  Document(page_content='气候变化：科学与模型。')],
 [Document(page_content='气候变化和经济影响。'),
  Document(page_content='气候变化对生物多样性的影响。'),
  Document(page_content='气候变化如何影响日常天气。'),
  Document(page_content='气候变化：社会视角。')],
 [Document(page_content='气候变化对公共健康的担忧。'),
  Document(page_content='气候变化和经济影响。'),
  Document(page_content='气候变化对生物多样性的影响。'),
  Document(page_content='气候变化如何影响日常天气。')]]

## Reciprocal Rank Fusion 算法

In [28]:
from langchain.load import dumps, loads


def reciprocal_rank_fusion(results: list[list], k=60):
    fused_scores = {}
    for docs in results:
        # 假设文档按相关性排序顺序返回
        for rank, doc in enumerate(docs):
            doc_str = dumps(doc)
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            previous_score = fused_scores[doc_str]
            fused_scores[doc_str] += 1 / (rank + k)

    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]
    return reranked_results

## 完整检索链

1. 生成一堆查询
2. 在检索器中查找每个查询
3. 使用倒数排名融合将所有结果连接在一起

In [29]:
chain = generate_queries | retriever.map() | reciprocal_rank_fusion

In [30]:
chain.invoke({"original_query": original_query})

D:\Software\anaconda3\envs\py310_LLMAppCourse2\lib\site-packages\langchain_core\_api\beta_decorator.py:87: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  warn_beta(


[(Document(page_content='气候变化和经济影响。'), 0.06612021857923497),
 (Document(page_content='气候变化对生物多样性的影响。'), 0.06531817380574652),
 (Document(page_content='气候变化如何影响日常天气。'), 0.06452452301209573),
 (Document(page_content='气候变化：社会视角。'), 0.031746031746031744),
 (Document(page_content='气候变化对公共健康的担忧。'), 0.016666666666666666),
 (Document(page_content='气候变化：科学与模型。'), 0.015873015873015872)]

# 优化Naive-RAG

## 索引

In [31]:
from dotenv import load_dotenv
from langchain import hub
from langchain_community.document_loaders import TextLoader
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 加载环境变量
load_dotenv()

# Load
file_path = "./data/translated_article.txt"
loader = TextLoader(file_path, encoding='utf-8')
docs = loader.load()

# Split
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
]
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on, strip_headers=False
)
markdown_text = "\n\n".join([doc.page_content for doc in docs])

md_header_splits = markdown_splitter.split_text(markdown_text)

chunk_size = 500
chunk_overlap = 50
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size, chunk_overlap=chunk_overlap
)

splits = text_splitter.split_documents(md_header_splits)

# Embed
embedding = OpenAIEmbeddings()

# Store
vectorstore = Chroma.from_documents(documents=splits, embedding=embedding)

# Retrieve
retriever = vectorstore.as_retriever()

## 生成多个查询语句

In [32]:
from langchain.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个有帮助的助手，会根据单个输入查询生成多个搜索查询。"),
    ("user", "生成与以下内容相关的多个搜索查询：{original_query}"),
    ("user", "输出（4 个查询）：")
])

generate_queries = (
    prompt | ChatOpenAI(temperature=0) | StrOutputParser() | (lambda x: x.split("\n"))
)

In [37]:
# test
question = "什么是few shot？"

generate_queries.invoke({"original_query": question})

['1. Few-shot learning 是什么意思？',
 '2. 如何在机器学习中使用 few-shot learning？',
 '3. Few-shot learning 的优势和局限性是什么？',
 '4. 什么是 few-shot classification 和 few-shot regression？']

## 在RAG中使用新的检索器

检索器的输入：str

检索器的输出：docs

### test 1

#### Reciprocal Rank Fusion 算法

In [38]:
from langchain.load import dumps, loads


def reciprocal_rank_fusion(results: list[list], k=60):
    fused_scores = {}
    for docs in results:
        # 假设文档按相关性排序顺序返回
        for rank, doc in enumerate(docs):
            doc_str = dumps(doc)
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            previous_score = fused_scores[doc_str]
            fused_scores[doc_str] += 1 / (rank + k)

    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]
    
    return reranked_results

#### 新的检索器

In [39]:
new_retriever = generate_queries | retriever.map() | reciprocal_rank_fusion

In [40]:
new_retriever.invoke({"original_query": original_query})

[(Document(page_content='气候变化和经济影响。'), 0.06612021857923497),
 (Document(page_content='气候变化对生物多样性的影响。'), 0.06531817380574652),
 (Document(page_content='气候变化如何影响日常天气。'), 0.06452452301209573),
 (Document(page_content='气候变化：社会视角。'), 0.031746031746031744),
 (Document(page_content='气候变化对公共健康的担忧。'), 0.016666666666666666),
 (Document(page_content='气候变化：科学与模型。'), 0.015873015873015872)]

In [42]:
new_retriever.invoke(original_query)

[(Document(page_content='气候变化和经济影响。'), 0.06612021857923497),
 (Document(page_content='气候变化对生物多样性的影响。'), 0.06531817380574652),
 (Document(page_content='气候变化如何影响日常天气。'), 0.06452452301209573),
 (Document(page_content='气候变化：社会视角。'), 0.031746031746031744),
 (Document(page_content='气候变化对公共健康的担忧。'), 0.016666666666666666),
 (Document(page_content='气候变化：科学与模型。'), 0.015873015873015872)]

#### 生成最终输出

In [41]:
# Generate
# prompt = hub.pull("rlm/rag-prompt")
system_prompt = (
    "你是一个用于问答任务的助手。"
    "使用以下检索到的上下文片段来回答问题。"
    "如果你不知道答案，就说你不知道。"
    "答案最多使用三句话，并保持简洁。"
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{question}"),
    ]
)
llm = ChatOpenAI()
output_parser = StrOutputParser()

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": new_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | output_parser
)

question = "什么是few shot？"
print(f"Q: {question}\nA: ")

result = rag_chain.invoke(question)
print(result)

Q: 什么是few shot？
A: 


AttributeError: 'tuple' object has no attribute 'page_content'

### test 2

#### Reciprocal Rank Fusion 算法

In [43]:
from langchain.load import dumps, loads


def reciprocal_rank_fusion(results: list[list], k=60):
    fused_scores = {}
    for docs in results:
        # 假设文档按相关性排序顺序返回
        for rank, doc in enumerate(docs):
            doc_str = dumps(doc)
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            previous_score = fused_scores[doc_str]
            fused_scores[doc_str] += 1 / (rank + k)

    # reranked_results = [
    #     (loads(doc), score)
    #     for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    # ]

    reranked_results = [
        loads(doc)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]
    
    return reranked_results

#### 新的检索器

In [44]:
new_retriever = generate_queries | retriever.map() | reciprocal_rank_fusion

In [45]:
new_retriever.invoke({"original_query": question})

[Document(page_content='# 基础提示  \n零样本（Zero-shot）和少样本（Few-shot）学习是两种基本的模型提示方法，这两种方法最早由多篇大语言模型论文提出，并广泛用于评估大语言模型的性能。', metadata={'Header 1': '基础提示'}),
 Document(page_content='## 少样本  \n**少样本学习**通过提供一系列高质量的示例，每个示例都包含目标任务的输入和期望输出，帮助模型更好地理解人类的意图和评价标准。因此，少样本学习通常比零样本学习有更好的表现。但这种方法会消耗更多的Token，并且当输入和输出文本较长时可能触及上下文长度的限制。  \n```\n文本: （劳伦斯在舞台上）到处跳跃，跳舞，跑步，出汗，擦脸，充分展现了他初次成名时的惊人才华。\n情感: 正面\n\n文本: 尽管所有迹象都表明相反，这部糟糕的电影还是设法冒充了一部正式的商业影片，收取全价门票，在电视上大肆宣传，自称能够娱乐小孩和成人。\n情感: 负面\n\n文本: 多年来第一次，德尼罗在情感上有了深刻的挖掘，可能是因为受到了合作伙伴出色表现的启发。\n情感: 正面', metadata={'Header 1': '基础提示', 'Header 2': '少样本'}),
 Document(page_content='文本: 我敢打赌视频游戏比电影有趣多了。\n情感:\n```  \n许多研究尝试了如何构建上下文示例以最大化性能，并发现**提示格式、训练示例及其顺序的选择可以极大地影响模型的性能**，从几乎是随机猜测到接近最优状态的表现。  \n[Zhao等人（2021年）](https://arxiv.org/abs/2102.09690)研究了少样本分类，并发现几种与LLM（他们使用的是GPT-3）相关的偏见导致了性能的高变异性：（1）*多数标签偏见*，如果示例中的标签分布不平衡；（2）*近期偏见*，模型可能倾向于重复最后出现的标签；（3）*常见Token偏见*，LLM倾向于生成常见的Token而不是罕见的Token。为了克服这些偏见，他们提出了一个方法，即当输入字符串为`N/A`时，调整模型输出的标签概率为均匀。', metadata={'Header 1': '基础提示', 'Header 2': '少样本'})

#### 生成最终输出

In [46]:
# Generate
# prompt = hub.pull("rlm/rag-prompt")
system_prompt = (
    "你是一个用于问答任务的助手。"
    "使用以下检索到的上下文片段来回答问题。"
    "如果你不知道答案，就说你不知道。"
    "答案最多使用三句话，并保持简洁。"
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{question}"),
    ]
)
llm = ChatOpenAI()
output_parser = StrOutputParser()

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": new_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | output_parser
)

question = "什么是few shot？"
print(f"Q: {question}\nA: ")

result = rag_chain.invoke(question)
print(result)

Q: 什么是few shot？
A: 
**Few-shot学习**是一种模型提示方法，通过提供少量高质量示例（通常是几个示例），每个示例包含输入和输出，以帮助模型更好地理解人类意图和评价标准。相较于零样本学习，Few-shot学习通常表现更好，但会消耗更多Token，并且可能受到上下文长度限制的影响。
